In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob

from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import Input, Dense, Reshape, Conv1DTranspose, BatchNormalization, Conv1D, LeakyReLU, Flatten, Lambda
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.losses import mse
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras import backend as K

In [2]:
from typing import List
from scipy.signal import butter, sosfiltfilt
import pandas as pd
import os
import numpy as np
from scipy.stats import zscore
from tqdm import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")



class PreprocessingEEG:
    def __init__(self, 
                 eeg_paths:List[str], 
                 lowcut=0.1, 
                 highcut=40, 
                 fs=250.0, 
                 order=4,
                 artifact_removal_zscore_amplitude=5,
                 artifact_removal_p2p=200,
                 segment_duration=50,
                 overlap=0.5
    ):
        """
        Args:                               
            eeg_paths (List[str]): Paths to EEG data files.
            lowcut (float): Lower cutoff frequency for bandpass filter.
            highcut (float): Upper cutoff frequency for bandpass filter.
            fs (float): Sampling frequency of the EEG data.
            order (int): Order of the Butterworth filter.
            artifact_removal_zscore_amplitude (float): Z-score threshold for artifact removal.
        """
        self.eeg_paths = eeg_paths
        self.lowcut = lowcut
        self.highcut = highcut
        self.fs = fs
        self.order = order
        self.artifact_removal_zscore_amplitude = artifact_removal_zscore_amplitude
        self.artifact_removal_p2p = artifact_removal_p2p
        self.segment_duration = segment_duration
        self.overlap = overlap

        self.clean_dfs = []
        self.sampling_datas_detail = []

    def bandpass_filter_scipy(self, data, highcut=None, lowcut=None, fs=None, order=None):
        """
        Applies a zero-phase Butterworth bandpass filter to EEG data.

        Args:
            data (np.array): 1D or 2D EEG signal (time series). 
                            If 2D, shape should be (n_channels, n_samples).
            lowcut (float): Lower cutoff frequency (Hz).
            highcut (float): Upper cutoff frequency (Hz).
            fs (float): Sampling frequency (Hz).
            order (int): Filter order (default=5).

        Returns:
            np.array: Bandpass-filtered EEG signal, same shape as input.
        """
        if lowcut is not None:
            self.lowcut = lowcut
        if highcut is not None:
            self.highcut = highcut  
        if fs is not None: 
            self.fs = fs
        if order is not None:
            self.order = order     
        print(f"[DEBUG] lowcut={self.lowcut}, highcut={self.highcut}, fs={self.fs}, order={self.order}")

        sos = butter(self.order, [self.lowcut, self.highcut], btype='band', fs=self.fs, output='sos')
        filtered = sosfiltfilt(sos, data, axis=-1)  # filter tiap channel
        return filtered
  
    def reject_epoch(self, epoch):
        # Peak-to-peak
        ptp = np.ptp(epoch)
        zscore_epoch = zscore(epoch)
        abs_zscore = np.abs(zscore_epoch)
        print(f"ptp: {ptp}, zscore abs: {np.max(abs_zscore)}")
        if ptp > self.artifact_removal_p2p:  # 200 µV, ubah ke Volt kalau datanya µV
            print("Rejecting due to peak-to-peak")
            return True
        
        # Z-score
        if np.any(abs_zscore) > self.artifact_removal_zscore_amplitude:
            print("Rejecting due to Z-score")
            return True
        
        return False
    
    def segment_signal(self,data):
        """
        Membagi sinyal menjadi segment dengan overlap.
        
        Args:
            data (np.array): 1D (n_samples) atau 2D (n_channels, n_samples).
            fs (int): Sampling rate dalam Hz.
            segment_duration (float): Panjang segmen dalam detik.
            overlap (float): Proporsi overlap (0–1), default 0.5 = 50%.
        
        Returns:
            np.array: Segment dengan shape (n_segments, n_channels, n_samples_per_segment).
        """
        print(f"Segmenting signal with shape: {data.shape}")
        if data.ndim == 1:
            data = data[np.newaxis, :]  # ubah ke (1, n_samples)

        # --- deteksi orientasi ---
        # asumsi: axis dengan panjang terbesar = waktu (samples)
        if data.shape[0] > data.shape[1]:
            # berarti (n_samples, n_channels), transpose jadi (n_channels, n_samples)
            data = data.T
            print("Transposed data to ensure (n_channels, n_samples)")
        print(f"After ensuring 2D, data shape: {data.shape}")
        n_channels, n_samples = data.shape
        seg_len = int(self.segment_duration * self.fs)
        step = int(seg_len * (1 - self.overlap))
        print(f"Segment length (samples): {seg_len}, Step size (samples): {step}")
        segments = []
        for start in range(0, n_samples - seg_len + 1, step):
            end = start + seg_len
            print(f"Creating segment from {start} to {end}")
            segments.append(data[:, start:end])
        print(f"Total segments created: {len(segments)}")
        return np.array(segments)  # shape = (n_segments, n_channels, seg_len)
    
    def listing_eeg_files(self,all_txt=True):
        list_data = []
        for root_path in self.eeg_paths:
            name_patient = os.listdir(root_path)
            for dir_patient in name_patient:
                label = "NORMAL" if "NORMAL" in root_path else "ASD"
                print(f"Processing patient: {dir_patient} with label: {label}")
                dir_patient_path = os.path.join(root_path, dir_patient)

                if os.path.isdir(dir_patient_path):
                    session_names = os.listdir(dir_patient_path)
                    if len(session_names) != 2 and all_txt==True:
                        print(f"Patient: {dir_patient}, Sessions: {session_names} skip")
                        continue
                    data = []
                    for session in session_names:
                        session_path = os.path.join(dir_patient_path, session)
                        if os.path.isdir(session_path):
                            txt_files = [f for f in os.listdir(session_path) if f.endswith(".txt")]
                            if len(txt_files) == 0:
                                print(f"No .txt files found in {session_path}, skip")
                                continue
                            if not all_txt:
                                list_path_txt = []
                                for txt_file in txt_files:
                                    list_path_txt.append(os.path.join(session_path, txt_file))
                                data.append(list_path_txt)
                            else:
                                data.append(os.path.join(session_path, txt_files[-1]))
                    if len(data) != 2 and all_txt==True:
                        print(f"Patient: {dir_patient} does not have 2 sessions, skip")
                        continue
                     
                    list_data.append((dir_patient,label, data))
        return list_data

    def load_and_preprocess(self,sampling=False,out_path=None,reject_epoch=True):
        list_data = self.listing_eeg_files()

        print(f"Total patients with 2 sessions: {len(list_data)}")

        labels = []
        datas = []
        clean_datasets = []
        total_epochs_rejected = 0
        total_epochs_accepted = 0
        data_status = []
        count_segments = defaultdict(int)

        for patient_idx, (patient_name,label, data_paths) in tqdm(enumerate(list_data)):
            print(f"Processing patient: {patient_name}")
            txt_1_path = data_paths[0]
            txt_2_path = data_paths[1]

            df1 = pd.read_csv(txt_1_path, comment="%", header=0)
            exg_cols_df_1 = [col for col in df1.columns if "EXG Channel" in col]
            exg_data_1 = df1[exg_cols_df_1]
            df2 = pd.read_csv(txt_2_path, comment="%", header=0)
            exg_cols_df_2 = [col for col in df2.columns if "EXG Channel" in col]
            exg_data_2 = df2[exg_cols_df_2]

            ch_names = ["Fp1","Fp2","F3","F4","C3","C4","P3","P4",
                    "O1","O2","F7","F8","T3","T4","T5","T6"]

            df_combine = pd.concat([exg_data_1, exg_data_2], axis=1)
            df_combine.columns = ch_names
           
            df_combine = df_combine.dropna().reset_index(drop=True)
            clean_data_combine_filtered = df_combine.dropna().reset_index(drop=True)
            print(f"After filtering drop na, data shape: {clean_data_combine_filtered.shape}")
            self.clean_dfs.append((patient_name,label, clean_data_combine_filtered))

            if sampling:
                epochs = self.segment_signal(clean_data_combine_filtered.values)
                epoch_accepted = 0
                print(f"Segmented into {len(epochs)} epochs of shape {epochs.shape[1:]} for patient {patient_name}")
                for idx, epoch in enumerate(epochs):
                    # apply bandpass filter to each channel
                    print(epoch.shape)
                    epoch = self.bandpass_filter_scipy(epoch)
                    print(f"After filtering bandpass_filter_scipy, data shape: {epoch.shape}")
                    epoch_df = pd.DataFrame(epoch.T, columns=ch_names)
                    count_segments[patient_name] += 1
                    if out_path:
                        out_segment_path = os.path.join(out_path,'segments')
                        out_patient_path = os.path.join(out_segment_path, patient_name)
                        if not os.path.exists(out_patient_path):
                            os.makedirs(out_patient_path)
                        

                    if self.reject_epoch(epoch) and reject_epoch:
                        filename_epoch = f"{patient_name}_epoch-{idx+1:05d}_rejected.csv"
                        print(f"Rejecting epoch {idx} for patient {patient_name} due to artifacts")
                        total_epochs_rejected += 1
                        if out_path:
                            epoch_df.to_csv(os.path.join(out_patient_path, filename_epoch), index=False)
                        continue
                    clean_datasets.append([patient_name,label, epoch])
                    if len(epoch) == 0:
                        continue
                    total_epochs_accepted += 1
                    labels.append(label)
                    datas.append(epoch)
                    epoch_accepted += 1
                    filename_epoch = f"{patient_name}_epoch-{idx+1:05d}_accepted.csv"
                    if out_path:
                        epoch_df.to_csv(os.path.join(out_patient_path, filename_epoch), index=False)

                data_status.append((patient_name, len(epochs), epoch_accepted))
    
            else:   
                clean_datasets.append([patient_name,label, clean_data_combine_filtered.values])
                labels.append(label)
                clean_data_combine_filtered = clean_data_combine_filtered.apply(lambda x: self.bandpass_filter_scipy(x))
                print(f"After filtering bandpass_filter_scipy, data shape: {clean_data_combine_filtered.shape}")
                datas.append(clean_data_combine_filtered.values.T)  # transpose ke (n_samples, n_channels)

        self.sampling_datas_detail = clean_datasets
        if sampling:
            print(f"Total epochs accepted: {total_epochs_accepted}, Total epochs rejected: {total_epochs_rejected}")
            print("Data status (patient, total epochs, accepted epochs):")
            for status in data_status:
                print(status)

            df_preprocessing_report = pd.DataFrame(count_segments.items(), columns=["Patient", "Num_Segments"])
            if out_path:
                df_preprocessing_report_path = os.path.join(out_path, "preprocessing_report.csv")
                df_preprocessing_report.to_csv(df_preprocessing_report_path, index=False)
                print(f"Preprocessing report saved to {df_preprocessing_report_path}")
        return labels, np.asanyarray(datas)